# TEMA + MACD BTC — Engine & Features

Run **all code cells once** (Runtime → Run all) to define indicators, ensemble backtest, and the research feature matrix.

Next notebook: `tema_macd_03_optimization.ipynb` or root `tema_macd_ensemble_btc_research.ipynb`.


In [ ]:
# Optional installs (uncomment if needed)
# !pip install pandas numpy yfinance matplotlib


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'strategies').exists() and (ROOT.parent / 'strategies').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Core engine (`tema_macd_ensemble_btc.py`)

In [ ]:
"""
TEMA + MACD ensemble for BTC-USD (daily).

- TEMA: classic triple-exponential moving average trend filter
- MACD: 12/26/9 momentum confirmation
- Ensemble: weighted vote (default 50/50) with optional consensus mode

Signals are shifted by one bar before P&L (no lookahead).
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import numpy as np
import pandas as pd

EnsembleMode = Literal["weighted", "consensus"]


def ema(series: pd.Series, span: int) -> pd.Series:
    return series.ewm(span=span, adjust=False).mean()


def tema(close: pd.Series, period: int) -> pd.Series:
    e1 = ema(close, period)
    e2 = ema(e1, period)
    e3 = ema(e2, period)
    return 3 * e1 - 3 * e2 + e3


def macd(
    close: pd.Series,
    fast: int = 12,
    slow: int = 26,
    signal: int = 9,
) -> pd.DataFrame:
    fast_ema = ema(close, fast)
    slow_ema = ema(close, slow)
    line = fast_ema - slow_ema
    sig = ema(line, signal)
    hist = line - sig
    return pd.DataFrame({"macd": line, "signal": sig, "hist": hist})


def tema_signal(close: pd.Series, tema_line: pd.Series) -> pd.Series:
    """+1 bullish, 0 neutral, -1 bearish."""
    above = close > tema_line
    slope = tema_line.diff() > 0
    sig = pd.Series(0.0, index=close.index)
    sig[above & slope] = 1.0
    sig[(~above) & (~slope)] = -1.0
    return sig


def macd_signal(m: pd.DataFrame) -> pd.Series:
    sig = pd.Series(0.0, index=m.index)
    sig[(m["hist"] > 0) & (m["macd"] > m["signal"])] = 1.0
    sig[(m["hist"] < 0) & (m["macd"] < m["signal"])] = -1.0
    return sig


@dataclass(frozen=True)
class EnsembleConfig:
    ticker: str = "BTC-USD"
    tema_period: int = 55
    macd_fast: int = 12
    macd_slow: int = 26
    macd_signal: int = 9
    tema_weight: float = 0.5
    macd_weight: float = 0.5
    mode: EnsembleMode = "weighted"
    long_only: bool = True
    long_threshold: float = 0.35
    flat_threshold: float = 0.05
    cost_bps: float = 10.0
    ann_days: int = 365


def combine_signals(
    s_tema: pd.Series,
    s_macd: pd.Series,
    cfg: EnsembleConfig,
) -> pd.Series:
    w_t = cfg.tema_weight
    w_m = cfg.macd_weight
    w_sum = w_t + w_m
    score = (w_t * s_tema + w_m * s_macd) / w_sum

    if cfg.mode == "consensus":
        pos = pd.Series(0.0, index=score.index)
        pos[(s_tema > 0) & (s_macd > 0)] = 1.0
        if not cfg.long_only:
            pos[(s_tema < 0) & (s_macd < 0)] = -1.0
        return pos

    pos = pd.Series(0.0, index=score.index)
    pos[score >= cfg.long_threshold] = 1.0
    if not cfg.long_only:
        pos[score <= -cfg.long_threshold] = -1.0
    # hysteresis band around zero
    pos[(score.abs() < cfg.flat_threshold)] = 0.0
    return pos


def build_ensemble_frame(close: pd.Series, cfg: EnsembleConfig) -> pd.DataFrame:
    t = tema(close, cfg.tema_period)
    m = macd(close, cfg.macd_fast, cfg.macd_slow, cfg.macd_signal)
    s_t = tema_signal(close, t)
    s_m = macd_signal(m)
    pos_raw = combine_signals(s_t, s_m, cfg)
    pos = pos_raw.shift(1).fillna(0.0)

    ret = close.pct_change().fillna(0.0)
    turnover = pos.diff().abs().fillna(0.0)
    strat_ret = pos * ret - turnover * (cfg.cost_bps / 1e4)
    equity = (1 + strat_ret).cumprod()

    out = pd.DataFrame(
        {
            "close": close,
            "tema": t,
            "macd": m["macd"],
            "macd_signal": m["signal"],
            "macd_hist": m["hist"],
            "sig_tema": s_t,
            "sig_macd": s_m,
            "ensemble_score": (cfg.tema_weight * s_t + cfg.macd_weight * s_m)
            / (cfg.tema_weight + cfg.macd_weight),
            "position": pos,
            "return": strat_ret,
            "equity": equity,
        },
        index=close.index,
    )
    return out


def max_drawdown(equity: pd.Series) -> float:
    peak = equity.cummax()
    dd = equity / peak - 1
    return float(dd.min())


def sharpe(returns: pd.Series, ann_days: int = 365) -> float:
    r = returns.dropna()
    if len(r) < 2 or r.std() == 0:
        return float("nan")
    return float(r.mean() / r.std() * np.sqrt(ann_days))


def summarize_backtest(df: pd.DataFrame, ann_days: int = 365) -> dict:
    eq = df["equity"].dropna()
    if len(eq) < 2:
        return {}
    tot = eq.iloc[-1] - 1
    years = len(df) / ann_days
    cagr = (eq.iloc[-1] ** (1 / max(years, 1e-9)) - 1) if years > 0 else float("nan")
    return {
        "total_return": float(tot),
        "cagr": float(cagr),
        "sharpe": sharpe(df["return"], ann_days),
        "max_drawdown": max_drawdown(eq),
        "bars": len(df),
    }


def load_btc_close(start: str = "2018-01-01") -> pd.Series:
    import yfinance as yf

    data = yf.download("BTC-USD", start=start, interval="1d", auto_adjust=True, progress=False)
    if data.empty:
        raise RuntimeError("Failed to download BTC-USD")
    if isinstance(data.columns, pd.MultiIndex):
        close = data["Close"]
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
    else:
        close = data["Close"]
    close = close.astype(float).squeeze()
    close.name = "BTC-USD"
    return close.dropna()


def run_default_ensemble(close: pd.Series | None = None) -> tuple[pd.DataFrame, dict]:
    cfg = EnsembleConfig()
    if close is None:
        close = load_btc_close()
    df = build_ensemble_frame(close, cfg)
    stats = summarize_backtest(df, cfg.ann_days)
    return df, stats


## Feature matrix (`tema_macd_features.py`)

In [ ]:
"""Feature library for TEMA/MACD BTC ensemble research."""

from __future__ import annotations

import numpy as np
import pandas as pd



def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    up = delta.clip(lower=0)
    down = (-delta).clip(lower=0)
    rs = up.ewm(alpha=1 / period, adjust=False).mean() / (
        down.ewm(alpha=1 / period, adjust=False).mean() + 1e-12
    )
    return 100 - (100 / (1 + rs))


def build_feature_matrix(
    close: pd.Series,
    tema_periods: tuple[int, ...] = (21, 34, 55, 89),
    macd_configs: tuple[tuple[int, int, int], ...] = ((8, 21, 5), (12, 26, 9), (19, 39, 9)),
) -> pd.DataFrame:
    feats: dict[str, pd.Series] = {}

    for p in tema_periods:
        t = tema(close, p)
        feats[f"tema_{p}_dist"] = (close / t - 1).replace([np.inf, -np.inf], np.nan)
        feats[f"tema_{p}_slope"] = t.pct_change(5)
        feats[f"sig_tema_{p}"] = tema_signal(close, t)

    for i, (fast, slow, sig) in enumerate(macd_configs):
        m = macd(close, fast, slow, sig)
        prefix = f"macd_{fast}_{slow}_{sig}"
        feats[f"{prefix}_hist"] = m["hist"]
        feats[f"{prefix}_hist_norm"] = m["hist"] / (close * 0.01 + 1e-9)
        feats[f"{prefix}_sig"] = macd_signal(m)
        feats[f"{prefix}_line"] = m["macd"]

    for w in (5, 10, 21, 63):
        feats[f"mom_{w}"] = close.pct_change(w)
        feats[f"vol_{w}"] = close.pct_change().rolling(w).std() * np.sqrt(365)

    feats["rsi_14"] = rsi(close, 14)
    feats["rsi_7"] = rsi(close, 7)

    ema20 = ema(close, 20)
    ema50 = ema(close, 50)
    ema200 = ema(close, 200)
    feats["ema20_50_spread"] = ema20 / ema50 - 1
    feats["ema50_200_spread"] = ema50 / ema200 - 1
    feats["above_ema200"] = (close > ema200).astype(float)

    df = pd.DataFrame(feats, index=close.index)
    return df.replace([np.inf, -np.inf], np.nan)


def forward_return_labels(close: pd.Series, horizon: int = 1) -> pd.Series:
    """Binary label: 1 if forward return > 0."""
    fwd = close.shift(-horizon) / close - 1
    return (fwd > 0).astype(int)


## Quick demo

In [ ]:
import matplotlib.pyplot as plt

close = load_btc_close('2018-01-01')
df, stats = run_default_ensemble(close)
print(stats)
print(df[['close','tema','macd_hist','ensemble_score','position']].tail())

plt.figure(figsize=(12,4))
plt.plot(df.index, df['equity'], label='Ensemble')
plt.plot(df.index, (1+close.pct_change().fillna(0)).cumprod(), alpha=0.5, label='BTC B&H')
plt.legend(); plt.title('TEMA+MACD ensemble'); plt.show()
